# Bi-stable double well: path overlap

Fractional overlap between each sampled trajectory and the subsequent trajectories of the Markov chain
(Fig. 2C of the main text), computed from the TPS simulations generated with
`bistable_double_well_generate_data.ipynb` (50 replicas per method).

For each reference path, the overlap with the following paths is tracked until no configuration is
shared any more. Overlaps are averaged over all reference paths of all replicas as a function of the
number of force evaluations spent after the reference path, and the average number of force evaluations
needed to renew a trajectory completely is computed (vertical lines in Fig. 2C).

| Output | Content |
|---|---|
| `output/<algorithm>/bsdw/<p_sel>/overlaps_<run>_all_replicas.pkl` | Overlap chains of all replicas (large, not tracked) |
| `results/<algorithm>/bsdw/<p_sel>/OPL_vs_feval_<run>.txt` | Average overlap vs force evaluations |
| `results/<algorithm>/bsdw/<p_sel>/OPL_renewal_<run>.txt` | Force evaluations for complete path renewal |

Computing the overlap chains takes several hours per method.

In [ ]:
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt

In [ ]:
from aaa_tps.utils.path_overlap import (compute_overlap_chain_1w, compute_overlap_chain_2w, flatten_overlaps,
                                        average_overlap_vs_force_evals, average_force_evals_to_zero_overlap)

In [ ]:
methods = {
    "2w_Gaus": dict(inputdir="./output/twoway_shooting/bsdw/Gaussian", outputdir="./results/twoway_shooting/bsdw/Gaussian",
                    run="2w", label="Two-Way (Gaussian)", chain=compute_overlap_chain_2w),
    "1w_Gaus": dict(inputdir="./output/oneway_shooting/bsdw/Gaussian", outputdir="./results/oneway_shooting/bsdw/Gaussian",
                    run="1w", label="One-Way (Gaussian)", chain=compute_overlap_chain_1w),
    "ARA_TPS_Unif": dict(inputdir="./output/ARA-TPS/bsdw/Uniform", outputdir="./results/ARA_TPS/bsdw/Uniform",
                         run="ARA-TPS", label="ARA-TPS (Uniform)", chain=compute_overlap_chain_1w),
    "ARA_TPS_Gaus": dict(inputdir="./output/ARA-TPS/bsdw/Gaussian", outputdir="./results/ARA_TPS/bsdw/Gaussian",
                         run="ARA-TPS", label="ARA-TPS (Gaussian)", chain=compute_overlap_chain_1w),
    "AAA_TPS_Unif": dict(inputdir="./output/AAA-TPS/bsdw/Uniform", outputdir="./results/AAA_TPS/bsdw/Uniform",
                         run="AAA-TPS", label="AAA-TPS (Uniform)", chain=compute_overlap_chain_1w),
    "AAA_TPS_Gaus": dict(inputdir="./output/AAA-TPS/bsdw/Gaussian", outputdir="./results/AAA_TPS/bsdw/Gaussian",
                         run="AAA-TPS", label="AAA-TPS (Gaussian)", chain=compute_overlap_chain_1w),
}

n_replicas = 50

for m in methods.values():
    os.makedirs(m["outputdir"], exist_ok=True)

## Overlap chains

The chains of each method are saved to `overlaps_<run>_all_replicas.pkl`, so this step can be skipped
when re-running the averages below.

In [ ]:
for key, m in methods.items():

    print(f"\n{m['label']}...")
    all_overlaps = []

    for replica in range(n_replicas):
        replica_dir = os.path.join(m["inputdir"], f"R_{replica:04d}")
        shooting_points = np.loadtxt(os.path.join(replica_dir, f"tot_sp_{m['run']}.txt"))
        with open(os.path.join(replica_dir, f"paths_{m['run']}.pkl"), 'rb') as f:
            trajectory_ensemble = pickle.load(f)

        all_overlaps.append(m["chain"](trajectory_ensemble, shooting_points))
        del trajectory_ensemble

    with open(os.path.join(m["inputdir"], f"overlaps_{m['run']}_all_replicas.pkl"), "wb") as f:
        pickle.dump(all_overlaps, f)

## Average overlap and complete renewal

The overlaps are averaged on a logarithmic grid of 200 force-evaluation values. Reference paths still
overlapping with the last path of their replica are censored and excluded from the average renewal.

In [ ]:
overlap_vs_feval = {}
renewal = {}

for key, m in methods.items():

    with open(os.path.join(m["inputdir"], f"overlaps_{m['run']}_all_replicas.pkl"), 'rb') as f:
        flat_overlaps = flatten_overlaps(pickle.load(f))

    force_eval = np.concatenate([np.loadtxt(os.path.join(m["inputdir"], f"R_{replica:04d}", f"force_evaluations_{m['run']}.txt"))
                                 for replica in range(n_replicas)])

    f_eval_opl, avg_overlap, _ = average_overlap_vs_force_evals(flat_overlaps, force_eval, spacing="log")
    overlap_vs_feval[key] = np.c_[f_eval_opl, avg_overlap]
    np.savetxt(os.path.join(m["outputdir"], f"OPL_vs_feval_{m['run']}.txt"), overlap_vs_feval[key])

    renewal[key] = average_force_evals_to_zero_overlap(flat_overlaps, force_eval)
    with open(os.path.join(m["outputdir"], f"OPL_renewal_{m['run']}.txt"), "w") as f:
        f.write("# Force evaluations needed for complete path renewal (overlap reaching zero)\n")
        f.write("# mean std n_decayed n_censored n_no_data\n")
        f.write(f"{renewal[key]['mean_fe_to_zero']:.10g} {renewal[key]['std_fe_to_zero']:.10g} "
                f"{renewal[key]['n_decayed']} {renewal[key]['n_censored']} {renewal[key]['n_no_data']}\n")

    print(f"{m['label']}: {renewal[key]['mean_fe_to_zero']:.1f} +- {renewal[key]['std_fe_to_zero']:.1f} "
          f"(n={renewal[key]['n_decayed']}, censored={renewal[key]['n_censored']}, no data={renewal[key]['n_no_data']})")

In [ ]:
fig, ax = plt.subplots(1, figsize=(10 * 0.39, 7.5 * 0.39), dpi=180)

for i, (key, m) in enumerate(methods.items()):
    ax.plot(overlap_vs_feval[key][:, 0], overlap_vs_feval[key][:, 1], color=f"C{i}", label=m["label"])
    ax.axvline(renewal[key]["mean_fe_to_zero"], color=f"C{i}", ls=":")

ax.axhline(y=0, color="k", ls="--", linewidth=1, alpha=0.5)
ax.set_xscale("log")
ax.set_xlim(1e2, 1e5)
ax.set_ylim(-0.1, 1)
ax.set_xlabel(r'Force Evaluations $n_{\text{F}}$')
ax.set_ylabel('Fractional Path Overlap')
ax.legend(frameon=False, fontsize=6)
plt.show()